In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Especificar opciones del Sampler

<Accordion>
<AccordionItem title="Versiones de paquetes">

El código de esta página fue desarrollado con los siguientes requisitos.
Recomendamos usar estas versiones o versiones más recientes.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Puedes usar opciones para personalizar la primitiva Sampler. Esta sección se centra en cómo especificar las opciones de la primitiva Qiskit Runtime. Aunque la interfaz del método `run()` de las primitivas es común a todas las implementaciones, sus opciones no lo son. Consulta las referencias de API correspondientes para obtener información sobre las opciones de [`qiskit.primitives.BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) y [`qiskit_aer.primitives.SamplerV2`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.primitives.SamplerV2.html).

<span id="pass-options"></span>
## Establecer opciones del Sampler
Puedes establecer opciones al inicializar el Sampler, después de inicializarlo, o actualizar las opciones una vez que el Sampler ha sido inicializado. Para instrucciones sobre cómo usar estas técnicas, consulta el tema [Introducción a las opciones](/guides/runtime-options-overview#options-precedence).

Además, puedes establecer el valor `shots` en el método `run()`, como se describe en la siguiente sección.
<span id="run-method"></span>
### Método Run()
Los únicos valores que puedes pasar a `run()` son los definidos en la interfaz, es decir, `shots`. Esto sobrescribe cualquier valor establecido para `default_shots` para la ejecución actual.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)

sampler = Sampler(mode=backend)
# Default shots to use if not specified in run()
sampler.options.default_shots = 500
# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d8286680bvlc73d1vmu0', 'sampler')>

### Special cases

<span id="shots"></span>
#### Shots

The `SamplerV2.run` method accepts two arguments: a list of PUBs, each of which can specify a PUB-specific value for shots, and a shots keyword argument. These shot values are a part of the Sampler execution interface, and are independent of the Runtime Sampler's options.  They take precedence over any values specified as options in order to comply with the Sampler abstraction.

However, if `shots` is not specified by any PUB or in the run keyword argument (or if they are all `None`), then the shots value from the options is used, most notably `default_shots`.

To summarize, this is the order of precedence for specifying shots in the Sampler, for any particular PUB:

1. If the PUB specifies shots, use that value.
2. If the `shots` keyword argument is specified in `run`, use that value.
4. If `twirling` is enabled  (True by default), then the product of `num_randomizations` and `shots_per_randomization`, as specified as  [`twirling` options](/docs/api/qiskit-ibm-runtime/options-twirling-options), is used.
5. If `sampler.options.default_shots` is specified, use that value.

Thus, if shots are specified in all possible places, the one with highest precedence (shots specified in the PUB) is used.

Example:

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)


# Setting shots during primitive initialization
sampler = Sampler(mode=backend, options={"default_shots": 4096})

# Setting options after primitive initialization
# This uses auto-complete.
sampler.options.default_shots = 2000

# This does bulk update.  The value for default_shots is overridden if you specify shots with run() or in the PUB.
sampler.options.update(
    default_shots=1024, dynamical_decoupling={"sequence_type": "XpXm"}
)

# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d82868ugbeec73alfa80', 'sampler')>

### Casos especiales
<span id="shots"></span>
#### Shots

El método `SamplerV2.run` acepta dos argumentos: una lista de PUBs, cada uno de los cuales puede especificar un valor de shots específico del PUB, y un argumento de palabra clave shots. Estos valores de shots forman parte de la interfaz de ejecución del Sampler y son independientes de las opciones del Sampler Runtime. Tienen prioridad sobre cualquier valor especificado como opción para cumplir con la abstracción del Sampler.

Sin embargo, si `shots` no está especificado por ningún PUB ni en el argumento de palabra clave run (o si todos son `None`), entonces se usa el valor de shots de las opciones, especialmente `default_shots`.

En resumen, este es el orden de precedencia para especificar shots en el Sampler, para cualquier PUB particular:

1. Si el PUB especifica shots, usar ese valor.
2. Si el argumento de palabra clave `shots` está especificado en `run`, usar ese valor.
4. Si `twirling` está habilitado (True por defecto), entonces se usa el producto de `num_randomizations` y `shots_per_randomization`, tal como se especifica en las [opciones de `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options).
5. Si `sampler.options.default_shots` está especificado, usar ese valor.

Por lo tanto, si los shots están especificados en todos los lugares posibles, se usa el de mayor precedencia (shots especificados en el PUB).

Ejemplo: